#Set up

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import get_colorscale

# Função do gráfico

In [ ]:
def plot_heatmap_dayofweek_hour(
    df_plot,
    titulo="Distribuição por dia da semana e hora",
    subtitulo_total=True,
    texto_subtitulo_total="Total de registros: {total}",
    insight=None,

    largura_px=1400,
    altura_px=900,
    template="plotly_white",
    cor_fundo="white",

    row_heights=(0.20, 0.80),
    column_widths=(0.78, 0.22),
    horizontal_spacing=0.05,
    vertical_spacing=0.08,

    colorscale_heatmap="RdBu_r",
    colorscale_barras="RdBu_r",
    inverter_escala_heatmap=False,
    inverter_escala_barras=False,

    usar_cor_variavel_barras=True,
    cor_barras_topo="#B8C4D6",
    cor_barras_lado="#B8C4D6",

    zmin_heatmap=None,
    zmax_heatmap=None,
    zmid_heatmap=None,

    mostrar_colorbar_heatmap=True,
    colorbar_title="Proporção",
    colorbar_tickformat=".1%",

    titulo_eixo_x_heatmap="Dia da semana",
    titulo_eixo_y_heatmap="Hora do dia",
    titulo_eixo_x_lado="Proporção",
    titulo_eixo_y_lado="",

    tamanho_fonte_titulo=24,
    tamanho_fonte_subtitulo=13,
    tamanho_fonte_insight=14,
    tamanho_fonte_ticks=11,
    tamanho_fonte_rotulos_barras=11,
    tamanho_fonte_rotulos_heatmap=10,

    cor_rotulos_barras_topo="#1f355e",
    cor_rotulos_barras_lado="#1f355e",

    cor_texto_claro="white",
    cor_texto_escuro="#1f355e",
    limiar_luminancia_texto=140,

    posicao_rotulos_topo="outside",
    posicao_rotulos_lado="outside",

    formato_rotulo_barras=".1%",
    formato_rotulo_heatmap=".1%",

    mostrar_rotulos_barras_topo=True,
    mostrar_rotulos_barras_lado=True,
    mostrar_rotulos_heatmap=True,

    usar_cor_texto_condicional_heatmap=True,
    mostrar_rotulo_heatmap_apenas_acima_de=None,

    margem_esquerda=70,
    margem_direita=90,
    margem_superior=120,
    margem_inferior=50,

    formatar_horas=True,
    sufixo_hora="h",
    mostrar_todos_horarios=False,
    intervalo_ticks_hora=2,
    rotacao_xticks=0,

    hover_dia="Dia: %{x}<br>Proporção: %{y:.1%}<extra></extra>",
    hover_heatmap="Dia: %{x}<br>Hora: %{y}<br>Proporção: %{z:.2%}<extra></extra>",
    hover_hora="Hora: %{y}<br>Proporção: %{x:.1%}<extra></extra>",

    exibir_figura=True,
):
    """
    Gera um gráfico composto com:
    1. Barras superiores por dia da semana
    2. Heatmap central por dia da semana e hora
    3. Barras laterais por hora

    Estrutura esperada em df_plot
    -----------------------------
    df_plot deve ser um dicionário com as chaves:
    - "matriz": DataFrame com index = horas e columns = dias da semana
    - "barras_topo": DataFrame com coluna "proporcao_total"
    - "barras_lado": DataFrame com coluna "proporcao_total"
    - "total_atendimentos": valor numérico total

    Recursos principais
    -------------------
    - Heatmap com escala reversível
    - Barras marginais com escala reversível
    - Rótulos do heatmap com cor automática conforme o fundo
    - Controle separado dos rótulos das barras
    - Exibição seletiva de horários
    - Insight opcional no título

    Retorno
    -------
    fig : plotly.graph_objects.Figure
        Figura final do Plotly.
    """

    # Importações locais para manter a função autocontida.
    # Isso facilita copiar e colar a função em outro notebook/script.
    import pandas as pd
    import numpy as np
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    from plotly.colors import get_colorscale

    # ==========================================================
    # 1. Funções auxiliares
    # ==========================================================
    # Este bloco reúne pequenas funções usadas internamente.
    # A ideia é deixar o corpo principal da função mais limpo
    # e separar pequenas responsabilidades.

    def inverter_colorscale(nome_escala):
        """
        Inverte uma escala nomeada do Plotly.
        Exemplo:
        RdBu_r -> RdBu
        RdBu -> RdBu_r
        """
        if not isinstance(nome_escala, str):
            return nome_escala
        return nome_escala[:-2] if nome_escala.endswith("_r") else f"{nome_escala}_r"

    def hex_para_rgb(hex_color):
        """
        Converte cor hexadecimal para tupla RGB.
        Exemplo:
        #FF0000 -> (255, 0, 0)
        """
        hex_color = hex_color.lstrip("#")
        return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

    def rgb_string_para_tuple(rgb_str):
        """
        Converte string rgb(...) ou hex para tupla RGB.
        """
        rgb_str = str(rgb_str).strip().lower()

        if rgb_str.startswith("#"):
            return hex_para_rgb(rgb_str)

        rgb_str = rgb_str.replace("rgb(", "").replace(")", "")
        partes = rgb_str.split(",")
        return tuple(int(float(p.strip())) for p in partes[:3])

    def obter_colorscale_plotly(nome_escala):
        """
        Retorna a colorscale do Plotly como lista de pares [posição, cor].
        """
        return get_colorscale(nome_escala)

    def interpolar_cor_colorscale(valor, vmin, vmax, colorscale):
        """
        Interpola a cor correspondente a um valor dentro de uma colorscale.
        Retorna tupla RGB.
        """
        # Se o valor estiver ausente, usa branco como fallback.
        # Isso evita erro em cálculos de cor para células vazias.
        if pd.isna(valor):
            return (255, 255, 255)

        # Normaliza o valor para a escala 0 a 1.
        # Se vmax == vmin, usamos 0.5 para evitar divisão por zero.
        if vmax == vmin:
            pos = 0.5
        else:
            pos = (valor - vmin) / (vmax - vmin)

        # Garante que a posição fique dentro do intervalo permitido.
        pos = max(0, min(1, pos))

        # Se a escala tiver só uma cor, retorna essa cor diretamente.
        if len(colorscale) == 1:
            return rgb_string_para_tuple(colorscale[0][1])

        # Busca o segmento da escala em que o valor se encaixa
        # e interpola a cor entre os dois pontos.
        for i in range(len(colorscale) - 1):
            p0, c0 = colorscale[i]
            p1, c1 = colorscale[i + 1]

            if p0 <= pos <= p1:
                rgb0 = rgb_string_para_tuple(c0)
                rgb1 = rgb_string_para_tuple(c1)

                if p1 == p0:
                    frac = 0
                else:
                    frac = (pos - p0) / (p1 - p0)

                r = round(rgb0[0] + frac * (rgb1[0] - rgb0[0]))
                g = round(rgb0[1] + frac * (rgb1[1] - rgb0[1]))
                b = round(rgb0[2] + frac * (rgb1[2] - rgb0[2]))

                return (r, g, b)

        # Fallback final caso o loop não encontre um intervalo.
        return rgb_string_para_tuple(colorscale[-1][1])

    def luminancia(rgb):
        """
        Calcula luminância percebida da cor.
        """
        r, g, b = rgb
        return 0.299 * r + 0.587 * g + 0.114 * b

    def escolher_cor_texto_por_fundo(valor, vmin, vmax, colorscale, cor_clara, cor_escura, limiar=140):
        """
        Define cor do texto com base no brilho da cor de fundo interpolada.
        """
        # Descobre a cor de fundo aproximada da célula
        # e escolhe texto claro ou escuro com base na luminância.
        cor_fundo_rgb = interpolar_cor_colorscale(valor, vmin, vmax, colorscale)
        return cor_clara if luminancia(cor_fundo_rgb) < limiar else cor_escura

    def formatar_hora(valor):
        """
        Formata hora para exibição.
        Exemplo:
        8 -> 08h
        """
        # Permite desligar a formatação, caso o eixo já venha pronto.
        if not formatar_horas:
            return str(valor)
        try:
            h = int(valor)
            return f"{h:02d}{sufixo_hora}"
        except:
            # Se não for possível converter, mantém o valor original.
            return str(valor)

    def selecionar_ticks_hora(horas_originais, horas_formatadas, mostrar_todos=False, intervalo=2):
        """
        Seleciona quais horários serão exibidos no eixo Y.
        """
        # Se o usuário quiser ver todos os horários, retorna tudo.
        if mostrar_todos:
            return horas_formatadas

        selecionados = []
        for i, h in enumerate(horas_originais):
            try:
                hh = int(h)
                if hh % intervalo == 0:
                    selecionados.append(horas_formatadas[i])
            except:
                # Se a hora não for numérica, usa o índice como fallback.
                if i % intervalo == 0:
                    selecionados.append(horas_formatadas[i])

        # Segurança: nunca retornar lista vazia.
        if len(selecionados) == 0:
            selecionados = horas_formatadas

        return selecionados

    # ==========================================================
    # 2. Ajuste de escala
    # ==========================================================
    # Permite inverter as escalas do heatmap e das barras de forma independente.
    # Isso é útil quando o usuário quer manter a mesma paleta,
    # mas mudar o sentido visual das cores.
    if inverter_escala_heatmap:
        colorscale_heatmap = inverter_colorscale(colorscale_heatmap)

    if inverter_escala_barras:
        colorscale_barras = inverter_colorscale(colorscale_barras)

    # ==========================================================
    # 3. Validações iniciais
    # ==========================================================
    # A função espera um dicionário com uma estrutura bem específica.
    # Aqui validamos a presença e o tipo de cada elemento essencial.
    if not isinstance(df_plot, dict):
        raise TypeError("O parâmetro 'df_plot' deve ser um dicionário.")

    chaves_necessarias = ["matriz", "barras_topo", "barras_lado", "total_atendimentos"]
    for chave in chaves_necessarias:
        if chave not in df_plot:
            raise ValueError(f"A chave '{chave}' não foi encontrada em df_plot.")

    matriz = df_plot["matriz"]
    barras_topo = df_plot["barras_topo"]
    barras_lado = df_plot["barras_lado"]
    total_atendimentos = df_plot["total_atendimentos"]

    if not isinstance(matriz, pd.DataFrame):
        raise TypeError("'matriz' deve ser um pandas.DataFrame.")

    if not isinstance(barras_topo, pd.DataFrame):
        raise TypeError("'barras_topo' deve ser um pandas.DataFrame.")

    if not isinstance(barras_lado, pd.DataFrame):
        raise TypeError("'barras_lado' deve ser um pandas.DataFrame.")

    if "proporcao_total" not in barras_topo.columns:
        raise ValueError("O DataFrame 'barras_topo' deve conter a coluna 'proporcao_total'.")

    if "proporcao_total" not in barras_lado.columns:
        raise ValueError("O DataFrame 'barras_lado' deve conter a coluna 'proporcao_total'.")

    if matriz.empty:
        raise ValueError("A 'matriz' está vazia.")

    # ==========================================================
    # 4. Preparação dos dados
    # ==========================================================
    # Faz cópias locais para evitar alterar os objetos originais.
    matriz_plot = matriz.copy()
    barras_topo_plot = barras_topo.copy()
    barras_lado_plot = barras_lado.copy()

    # Extrai os eixos principais do heatmap.
    dias = list(matriz_plot.columns)
    horas = list(matriz_plot.index)

    # Converte os dados para arrays numéricos.
    # z = valores do heatmap
    # totais_dia = barras superiores
    # totais_hora = barras laterais
    z = matriz_plot.astype(float).values
    totais_dia = barras_topo_plot["proporcao_total"].astype(float).values
    totais_hora = barras_lado_plot["proporcao_total"].astype(float).values

    # Garante coerência entre dimensões da matriz e das barras marginais.
    if len(dias) != len(totais_dia):
        raise ValueError("O número de dias em 'matriz' precisa coincidir com o tamanho de 'barras_topo'.")

    if len(horas) != len(totais_hora):
        raise ValueError("O número de horas em 'matriz' precisa coincidir com o tamanho de 'barras_lado'.")

    # Calcula máximos e mínimos usados em escala e limites visuais.
    max_topo = np.nanmax(totais_dia) if len(totais_dia) > 0 else 0
    max_lado = np.nanmax(totais_hora) if len(totais_hora) > 0 else 0
    max_heatmap = np.nanmax(z) if z.size > 0 else 0
    min_heatmap = np.nanmin(z) if z.size > 0 else 0

    # Se o usuário não informar os limites do heatmap,
    # a função usa os próprios limites dos dados.
    if zmin_heatmap is None:
        zmin_heatmap = min_heatmap

    if zmax_heatmap is None:
        zmax_heatmap = max_heatmap if max_heatmap > zmin_heatmap else zmin_heatmap + 1e-9

    # Gera os rótulos formatados das horas.
    horas_texto = [formatar_hora(h) for h in horas]

    # Define quais horários realmente serão mostrados nos eixos.
    # Isso ajuda quando existem muitas linhas e o eixo ficaria poluído.
    ticks_hora_mostrar = selecionar_ticks_hora(
        horas_originais=horas,
        horas_formatadas=horas_texto,
        mostrar_todos=mostrar_todos_horarios,
        intervalo=intervalo_ticks_hora
    )

    # Pré-formata os textos das barras.
    textos_topo = [format(v, formato_rotulo_barras) for v in totais_dia]
    textos_lado = [format(v, formato_rotulo_barras) for v in totais_hora]

    # ==========================================================
    # 5. Preparação dos rótulos do heatmap
    # ==========================================================
    # Recupera a escala real do Plotly em formato utilizável para interpolação.
    colorscale_heatmap_real = obter_colorscale_plotly(colorscale_heatmap)

    # text_heatmap guarda o texto visível em cada célula.
    # textfont_colors guarda a cor ideal do texto em cada célula.
    text_heatmap = []
    textfont_colors = []

    for i in range(z.shape[0]):
        linha_texto = []
        linha_cor = []

        for j in range(z.shape[1]):
            valor = z[i, j]

            # Se o valor é ausente, não mostra texto.
            if pd.isna(valor):
                linha_texto.append("")
                linha_cor.append(cor_texto_escuro)
                continue

            # Define se o valor deve ou não aparecer como rótulo.
            # Pode ser útil esconder valores muito pequenos.
            if (
                mostrar_rotulo_heatmap_apenas_acima_de is not None
                and valor < mostrar_rotulo_heatmap_apenas_acima_de
            ):
                linha_texto.append("")
            else:
                linha_texto.append(format(valor, formato_rotulo_heatmap))

            # Escolhe a cor do texto com base na cor do fundo da célula.
            # Isso melhora a legibilidade em células escuras e claras.
            if usar_cor_texto_condicional_heatmap:
                cor_final = escolher_cor_texto_por_fundo(
                    valor=valor,
                    vmin=zmin_heatmap,
                    vmax=zmax_heatmap,
                    colorscale=colorscale_heatmap_real,
                    cor_clara=cor_texto_claro,
                    cor_escura=cor_texto_escuro,
                    limiar=limiar_luminancia_texto
                )
            else:
                cor_final = cor_texto_escuro

            linha_cor.append(cor_final)

        text_heatmap.append(linha_texto)
        textfont_colors.append(linha_cor)

    # ==========================================================
    # 6. Construção do subplot
    # ==========================================================
    # O gráfico final é uma composição 2x2:
    # superior esquerdo = barras por dia
    # inferior esquerdo = heatmap principal
    # inferior direito = barras por hora
    # superior direito = célula vazia
    fig = make_subplots(
        rows=2,
        cols=2,
        row_heights=list(row_heights),
        column_widths=list(column_widths),
        horizontal_spacing=horizontal_spacing,
        vertical_spacing=vertical_spacing,
        specs=[
            [{"type": "bar"}, {"type": "xy"}],
            [{"type": "heatmap"}, {"type": "bar"}]
        ]
    )

    # ==========================================================
    # 7. Barras do topo
    # ==========================================================
    # Permite duas estratégias:
    # 1. cor variável conforme o valor
    # 2. uma cor fixa para todas as barras
    marker_topo = (
        dict(
            color=totais_dia,
            colorscale=colorscale_barras,
            cmin=0,
            cmax=max_topo if max_topo > 0 else 1
        )
        if usar_cor_variavel_barras
        else dict(color=cor_barras_topo)
    )

    fig.add_trace(
        go.Bar(
            x=dias,
            y=totais_dia,
            marker=marker_topo,
            showlegend=False,
            hovertemplate=hover_dia,
            text=textos_topo if mostrar_rotulos_barras_topo else None,
            textposition=posicao_rotulos_topo if mostrar_rotulos_barras_topo else None,
            textfont=dict(
                size=tamanho_fonte_rotulos_barras,
                color=cor_rotulos_barras_topo
            ),
            cliponaxis=False
        ),
        row=1, col=1
    )

    # ==========================================================
    # 8. Heatmap central
    # ==========================================================
    # Esse é o componente principal do gráfico.
    # Ele mostra a distribuição cruzada entre dia da semana e hora.
    heatmap_trace = go.Heatmap(
        z=z,
        x=dias,
        y=horas_texto if formatar_horas else horas,
        colorscale=colorscale_heatmap,
        zmin=zmin_heatmap,
        zmax=zmax_heatmap,
        zmid=zmid_heatmap,
        showscale=mostrar_colorbar_heatmap,
        colorbar=dict(
            title=colorbar_title,
            tickformat=colorbar_tickformat
        ) if mostrar_colorbar_heatmap else None,
        hovertemplate=hover_heatmap
    )

    fig.add_trace(heatmap_trace, row=2, col=1)

    # Como o Heatmap do Plotly não permite cor individual de texto por célula
    # de forma direta, usamos annotations sobre cada célula.
    if mostrar_rotulos_heatmap:
        for i, hora_label in enumerate(horas_texto if formatar_horas else horas):
            for j, dia_label in enumerate(dias):
                texto = text_heatmap[i][j]
                if texto == "":
                    continue

                fig.add_annotation(
                    x=dia_label,
                    y=hora_label,
                    text=texto,
                    showarrow=False,
                    font=dict(
                        size=tamanho_fonte_rotulos_heatmap,
                        color=textfont_colors[i][j]
                    ),
                    xref="x3",
                    yref="y3"
                )

    # ==========================================================
    # 9. Barras laterais
    # ==========================================================
    # Mesma lógica das barras do topo, mas na horizontal.
    marker_lado = (
        dict(
            color=totais_hora,
            colorscale=colorscale_barras,
            cmin=0,
            cmax=max_lado if max_lado > 0 else 1
        )
        if usar_cor_variavel_barras
        else dict(color=cor_barras_lado)
    )

    fig.add_trace(
        go.Bar(
            x=totais_hora,
            y=horas_texto if formatar_horas else horas,
            orientation="h",
            marker=marker_lado,
            showlegend=False,
            hovertemplate=hover_hora,
            text=textos_lado if mostrar_rotulos_barras_lado else None,
            textposition=posicao_rotulos_lado if mostrar_rotulos_barras_lado else None,
            textfont=dict(
                size=tamanho_fonte_rotulos_barras,
                color=cor_rotulos_barras_lado
            ),
            cliponaxis=False
        ),
        row=2, col=2
    )

    # ==========================================================
    # 10. Título
    # ==========================================================
    # Monta dinamicamente o título em blocos:
    # - título principal
    # - insight opcional
    # - subtítulo com total
    total_formatado = f"{int(total_atendimentos):,}".replace(",", ".")

    blocos_titulo = []
    if titulo and str(titulo).strip():
        blocos_titulo.append(f"<b>{titulo}</b>")

    if insight is not None and str(insight).strip() != "":
        blocos_titulo.append(
            f"<span style='font-size:{tamanho_fonte_insight}px'>{insight}</span>"
        )

    if subtitulo_total:
        blocos_titulo.append(
            f"<sup style='font-size:{tamanho_fonte_subtitulo}px'>"
            f"{texto_subtitulo_total.format(total=total_formatado)}"
            f"</sup>"
        )

    titulo_final = "<br>".join(blocos_titulo)

    # ==========================================================
    # 11. Layout geral
    # ==========================================================
    # Ajustes gerais de tamanho, fundo, margens e posição do título.
    fig.update_layout(
        title=dict(
            text=titulo_final,
            x=0.5,
            xanchor="center",
            y=0.98,
            yanchor="top",
            font=dict(size=tamanho_fonte_titulo)
        ),
        template=template,
        width=largura_px,
        height=altura_px,
        paper_bgcolor=cor_fundo,
        plot_bgcolor=cor_fundo,
        margin=dict(
            l=margem_esquerda,
            r=margem_direita,
            t=margem_superior,
            b=margem_inferior
        ),
        bargap=0.18
    )

    # ==========================================================
    # 12. Eixos do topo
    # ==========================================================
    # No painel superior, mostramos apenas o essencial.
    # O eixo Y é ocultado para deixar a leitura mais limpa.
    fig.update_xaxes(
        row=1, col=1,
        showgrid=False,
        tickfont=dict(size=tamanho_fonte_ticks),
        tickangle=rotacao_xticks,
        title=""
    )
    fig.update_yaxes(
        row=1, col=1,
        showgrid=False,
        showticklabels=False,
        zeroline=False,
        title=""
    )

    # ==========================================================
    # 13. Eixos do heatmap
    # ==========================================================
    # O eixo Y do heatmap é invertido para manter a leitura mais intuitiva,
    # com horários iniciais no topo e finais abaixo, se essa for a intenção visual.
    fig.update_xaxes(
        row=2, col=1,
        showgrid=False,
        tickfont=dict(size=tamanho_fonte_ticks),
        tickangle=rotacao_xticks,
        title=titulo_eixo_x_heatmap
    )
    fig.update_yaxes(
        row=2, col=1,
        autorange="reversed",
        title=titulo_eixo_y_heatmap,
        showgrid=False,
        tickfont=dict(size=tamanho_fonte_ticks),
        categoryorder="array",
        categoryarray=horas_texto
    )

    # ==========================================================
    # 14. Eixos da barra lateral
    # ==========================================================
    # O eixo X da barra lateral usa percentual.
    # Também deixamos uma folga extra à direita para não cortar rótulos.
    fig.update_xaxes(
        row=2, col=2,
        tickformat=".0%",
        showgrid=False,
        zeroline=False,
        title=titulo_eixo_x_lado,
        tickfont=dict(size=tamanho_fonte_ticks),
        range=[0, (max_lado * 1.22) if max_lado > 0 else 1]
    )
    fig.update_yaxes(
        row=2, col=2,
        autorange="reversed",
        title=titulo_eixo_y_lado,
        showgrid=False,
        tickfont=dict(size=tamanho_fonte_ticks),
        categoryorder="array",
        categoryarray=horas_texto
    )

    # Oculta o quadrante superior direito, que existe apenas por causa da grade 2x2.
    fig.update_xaxes(row=1, col=2, visible=False)
    fig.update_yaxes(row=1, col=2, visible=False)

    # ==========================================================
    # 15. Limpeza visual dos horários
    # ==========================================================
    # Aplica exatamente o mesmo subconjunto de horários
    # nos dois eixos Y inferiores, mantendo alinhamento visual.
    fig.update_yaxes(
        row=2, col=1,
        tickmode="array",
        tickvals=ticks_hora_mostrar,
        ticktext=ticks_hora_mostrar
    )
    fig.update_yaxes(
        row=2, col=2,
        tickmode="array",
        tickvals=ticks_hora_mostrar,
        ticktext=ticks_hora_mostrar
    )

    # ==========================================================
    # 16. Espaço extra para rótulos do topo
    # ==========================================================
    # Se os rótulos das barras de cima estiverem "outside",
    # criamos folga extra no eixo Y para não cortar o texto.
    if mostrar_rotulos_barras_topo and posicao_rotulos_topo == "outside":
        fig.update_yaxes(
            row=1, col=1,
            range=[0, (max_topo * 1.18) if max_topo > 0 else 1]
        )

    # ==========================================================
    # 17. Exibição
    # ==========================================================
    # Exibe a figura automaticamente, se solicitado.
    if exibir_figura:
        fig.show()

    return fig

# Dados de demostração

In [ ]:
# ==========================================================
# 1. Base tabular, como se viesse de uma query
# ==========================================================
df_base_teste = pd.DataFrame({
    "dia_semana": [
        "Seg","Ter","Qua","Qui","Sex","Sáb","Dom",
        "Seg","Ter","Qua","Qui","Sex","Sáb","Dom",
        "Seg","Ter","Qua","Qui","Sex","Sáb","Dom",
        "Seg","Ter","Qua","Qui","Sex","Sáb","Dom",
        "Seg","Ter","Qua","Qui","Sex","Sáb","Dom",
        "Seg","Ter","Qua","Qui","Sex","Sáb","Dom"
    ],
    "hora": [
        8,8,8,8,8,8,8,
        9,9,9,9,9,9,9,
        10,10,10,10,10,10,10,
        11,11,11,11,11,11,11,
        14,14,14,14,14,14,14,
        15,15,15,15,15,15,15
    ],
    "quantidade": [
        820, 790, 810, 805, 830, 420, 460,
        910, 880, 940, 900, 950, 500, 560,
        980, 930, 960, 990, 970, 540, 590,
        920, 890, 910, 940, 930, 510, 550,
        700, 680, 720, 740, 710, 430, 410,
        690, 670, 710, 730, 700, 420, 400
    ]
})

# ==========================================================
# 2. Criar grade completa de dia x hora
# ==========================================================
dias_semana = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]
horas = list(range(24))

grade_completa = pd.MultiIndex.from_product(
    [dias_semana, horas],
    names=["dia_semana", "hora"]
).to_frame(index=False)

df_base_teste = grade_completa.merge(
    df_base_teste,
    on=["dia_semana", "hora"],
    how="left"
)

# ==========================================================
# 3. Preencher horários ausentes com padrão compreensível
# ==========================================================
mapa_quantidade_padrao = {
    0: 120, 1: 110, 2: 100, 3: 95, 4: 105, 5: 110, 6: 130, 7: 160,
    8: None, 9: None, 10: None, 11: None,
    12: 140,
    13: 520, 14: None, 15: None, 16: 540, 17: 500,
    18: 240, 19: 250, 20: 230, 21: 220, 22: 210, 23: 200
}

ajuste_fim_semana = {
    "Seg": 1.00,
    "Ter": 0.98,
    "Qua": 1.02,
    "Qui": 1.03,
    "Sex": 1.01,
    "Sáb": 0.72,
    "Dom": 0.75
}

def preencher_quantidade(row):
    if pd.notna(row["quantidade"]):
        return row["quantidade"]

    base = mapa_quantidade_padrao[row["hora"]]
    return round(base * ajuste_fim_semana[row["dia_semana"]])

df_base_teste["quantidade"] = df_base_teste.apply(preencher_quantidade, axis=1)

# ==========================================================
# 4. Calcular proporção
# ==========================================================
total_atendimentos = df_base_teste["quantidade"].sum()
df_base_teste["proporcao"] = df_base_teste["quantidade"] / total_atendimentos

# ==========================================================
# 5. Matriz do heatmap
# ==========================================================
matriz_teste = df_base_teste.pivot(
    index="hora",
    columns="dia_semana",
    values="proporcao"
).reindex(index=horas, columns=dias_semana)

# ==========================================================
# 6. Barras do topo
# ==========================================================
barras_topo_teste = (
    df_base_teste
    .groupby("dia_semana", as_index=False)["quantidade"]
    .sum()
)

barras_topo_teste["proporcao_total"] = (
    barras_topo_teste["quantidade"] / barras_topo_teste["quantidade"].sum()
)

barras_topo_teste = (
    barras_topo_teste
    .set_index("dia_semana")
    .reindex(dias_semana)
    .reset_index()
)

# ==========================================================
# 7. Barras laterais
# ==========================================================
barras_lado_teste = (
    df_base_teste
    .groupby("hora", as_index=False)["quantidade"]
    .sum()
)

barras_lado_teste["proporcao_total"] = (
    barras_lado_teste["quantidade"] / barras_lado_teste["quantidade"].sum()
)

barras_lado_teste = (
    barras_lado_teste
    .set_index("hora")
    .reindex(horas)
    .reset_index()
)

# ==========================================================
# 8. Estrutura final
# ==========================================================
df_plot_teste = {
    "matriz": matriz_teste,
    "barras_topo": barras_topo_teste,
    "barras_lado": barras_lado_teste,
    "total_atendimentos": total_atendimentos
}

# Visualizar a base principal
display(df_base_teste.head(20))

,dia_semana,hora,quantidade,proporcao
0,Seg,0,120.0,0.002113
1,Seg,1,110.0,0.001937
2,Seg,2,100.0,0.001761
3,Seg,3,95.0,0.001672
4,Seg,4,105.0,0.001849
5,Seg,5,110.0,0.001937
6,Seg,6,130.0,0.002289
7,Seg,7,160.0,0.002817
8,Seg,8,820.0,0.014436
9,Seg,9,910.0,0.016021


# Plotagem

In [ ]:
fig = plot_heatmap_dayofweek_hour(
    df_plot=df_plot_teste,
    titulo="Distribuição por dia da semana e hora",
    insight="Maior concentração entre 08h e 11h nos dias úteis",
    texto_subtitulo_total="Total de registros: {total}",

    largura_px=1400,
    altura_px=900,

    colorscale_heatmap="RdBu_r",
    colorscale_barras="RdBu_r",
    inverter_escala_heatmap=False,
    inverter_escala_barras=False,

    usar_cor_variavel_barras=True,

    zmid_heatmap=df_plot_teste["matriz"].values.mean(),
    mostrar_colorbar_heatmap=True,
    colorbar_title="Proporção",
    colorbar_tickformat=".1%",

    mostrar_rotulos_barras_topo=True,
    mostrar_rotulos_barras_lado=True,
    mostrar_rotulos_heatmap=True,

    posicao_rotulos_topo="outside",
    posicao_rotulos_lado="outside",

    cor_rotulos_barras_topo="#1f355e",
    cor_rotulos_barras_lado="#1f355e",

    formato_rotulo_barras=".1%",
    formato_rotulo_heatmap=".1%",

    usar_cor_texto_condicional_heatmap=True,
    mostrar_rotulo_heatmap_apenas_acima_de=None,
    tamanho_fonte_rotulos_heatmap=9,

    titulo_eixo_x_heatmap="Dia da semana",
    titulo_eixo_y_heatmap="Hora do dia",
    titulo_eixo_x_lado="Proporção",

    formatar_horas=True,
    mostrar_todos_horarios=False,
    intervalo_ticks_hora=2
)